# Iowa Liquor Sales: Data Exploration & Analysis
## SQL Analytics with BigQuery

End-to-end SQL analysis of the Iowa Liquor Sales public dataset, covering data profiling, sales analytics, store performance, and data quality.

**Dataset**: `bigquery-public-data.iowa_liquor_sales.sales` (~33.6M rows, 2012-present)

### Contents:
1. Data Exploration
2. Sales Overview
3. Store Performance
4. Product & Category Analysis
5. Time Series & Trends
6. Data Quality
7. SQL Quick Reference

---
## Setup & BigQuery Connection

In [1]:
import pandas as pd
import numpy as np
import os
from google.cloud import bigquery
from google.oauth2 import service_account

pd.options.display.float_format = '{:,.2f}'.format
pd.options.styler.format.formatter = '{:,}'.format

credentials = service_account.Credentials.from_service_account_file('../service_account_iowa.json')
client = bigquery.Client(credentials=credentials, project=credentials.project_id)

def run_sql(file):
    with open(f'../sql_queries/{file}') as f:
        return client.query(f.read()).to_dataframe()

def run_query(sql):
    return client.query(sql).to_dataframe()

### Quick dataset preview

In [3]:
# Quick peek - transposed so we can see all columns
run_query("""
    SELECT *
    FROM `bigquery-public-data.iowa_liquor_sales.sales`
    LIMIT 2
""").T

,0,1
invoice_and_item_number,RINV-05963200013,RINV-06039200126
date,2025-10-28,2025-12-29
store_number,2561,4988
store_name,HY-VEE FOOD STORE (1148) / FLEUR / DSM,HAPPY'S WINE & SPIRITS
address,4605 FLEUR DRIVE,5925 UNIVERSITY AVE
city,DES MOINES,CEDAR FALLS
zip_code,50321.0,50613.0
store_location,POINT(-93.64348 41.5422),POINT(-92.42849 42.51302)
county_number,None,None
county,POLK,BLACK HAWK


In [4]:
# Dataset overview
run_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT store_number) AS unique_stores,
        COUNT(DISTINCT item_description) AS unique_products,
        COUNT(DISTINCT city) AS unique_cities,
        MIN(date) AS earliest_date,
        MAX(date) AS latest_date,
        ROUND(SUM(sale_dollars), 2) AS total_revenue
    FROM `bigquery-public-data.iowa_liquor_sales.sales`
""")


,total_rows,unique_stores,unique_products,unique_cities,earliest_date,latest_date,total_revenue
0,33615130,3398,14371,504,2012-01-03,2026-02-27,"4,996,270,526.34"


---
## 1. Data Exploration

Profiling the full dataset (~33.6M rows) using SQL directly in BigQuery for scalability.


In [6]:
# Column types across the dataset
run_query("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM `bigquery-public-data.iowa_liquor_sales`.INFORMATION_SCHEMA.COLUMNS
    WHERE table_name = 'sales'
    ORDER BY ordinal_position
""")


,column_name,data_type,is_nullable
0,invoice_and_item_number,STRING,YES
1,date,DATE,YES
2,store_number,STRING,YES
3,store_name,STRING,YES
4,address,STRING,YES
5,city,STRING,YES
6,zip_code,STRING,YES
7,store_location,GEOGRAPHY,YES
8,county_number,STRING,YES
9,county,STRING,YES


In [7]:
# Null count per column
run_query("""
    SELECT
        COUNTIF(invoice_and_item_number IS NULL) AS invoice_nulls,
        COUNTIF(date IS NULL) AS date_nulls,
        COUNTIF(store_number IS NULL) AS store_number_nulls,
        COUNTIF(store_name IS NULL) AS store_name_nulls,
        COUNTIF(address IS NULL) AS address_nulls,
        COUNTIF(city IS NULL) AS city_nulls,
        COUNTIF(zip_code IS NULL) AS zip_code_nulls,
        COUNTIF(county IS NULL) AS county_nulls,
        COUNTIF(category IS NULL) AS category_nulls,
        COUNTIF(category_name IS NULL) AS category_name_nulls,
        COUNTIF(vendor_number IS NULL) AS vendor_number_nulls,
        COUNTIF(vendor_name IS NULL) AS vendor_name_nulls,
        COUNTIF(item_number IS NULL) AS item_number_nulls,
        COUNTIF(item_description IS NULL) AS item_description_nulls,
        COUNTIF(pack IS NULL) AS pack_nulls,
        COUNTIF(bottle_volume_ml IS NULL) AS bottle_volume_nulls,
        COUNTIF(state_bottle_cost IS NULL) AS state_bottle_cost_nulls,
        COUNTIF(state_bottle_retail IS NULL) AS state_bottle_retail_nulls,
        COUNTIF(bottles_sold IS NULL) AS bottles_sold_nulls,
        COUNTIF(sale_dollars IS NULL) AS sale_dollars_nulls,
        COUNTIF(volume_sold_liters IS NULL) AS volume_sold_liters_nulls,
        COUNTIF(volume_sold_gallons IS NULL) AS volume_sold_gallons_nulls
    FROM `bigquery-public-data.iowa_liquor_sales.sales`
""").T.rename(columns={0: "null_count"})


,null_count
invoice_nulls,0
date_nulls,0
store_number_nulls,0
store_name_nulls,0
address_nulls,85273
city_nulls,85272
zip_code_nulls,85339
county_nulls,162075
category_nulls,16974
category_name_nulls,25040


In [12]:
# Numerical statistics for key columns
run_query("""
    SELECT
        COUNT(*) AS total_rows,
        ROUND(AVG(sale_dollars), 2) AS avg_sale_dollars,
        ROUND(STDDEV(sale_dollars), 2) AS std_sale_dollars,
        ROUND(MIN(sale_dollars), 2) AS min_sale_dollars,
        ROUND(MAX(sale_dollars), 2) AS max_sale_dollars,
        ROUND(AVG(bottles_sold), 2) AS avg_bottles_sold,
        ROUND(STDDEV(bottles_sold), 2) AS std_bottles_sold,
        MIN(bottles_sold) AS min_bottles_sold,
        MAX(bottles_sold) AS max_bottles_sold,
        ROUND(AVG(volume_sold_liters), 2) AS avg_volume_liters,
        ROUND(STDDEV(volume_sold_liters), 2) AS std_volume_liters,
        ROUND(MIN(volume_sold_liters), 2) AS min_volume_liters,
        ROUND(MAX(volume_sold_liters), 2) AS max_volume_liters
    FROM `bigquery-public-data.iowa_liquor_sales.sales`
""").T


,0
total_rows,33615130
avg_sale_dollars,148.63
std_sale_dollars,526.33
min_sale_dollars,"-9,720.00"
max_sale_dollars,"279,557.28"
avg_bottles_sold,11.01
std_bottles_sold,31.02
min_bottles_sold,-768
max_bottles_sold,15000
avg_volume_liters,9.12


In [14]:
# Categorical column cardinality
run_query("""
    SELECT
        COUNT(DISTINCT store_name) AS unique_stores,
        COUNT(DISTINCT city) AS unique_cities,
        COUNT(DISTINCT county) AS unique_counties,
        COUNT(DISTINCT category_name) AS unique_categories,
        COUNT(DISTINCT vendor_name) AS unique_vendors,
        COUNT(DISTINCT item_description) AS unique_items
    FROM `bigquery-public-data.iowa_liquor_sales.sales`
""")


,unique_stores,unique_cities,unique_counties,unique_categories,unique_vendors,unique_items
0,3659,504,100,103,660,14371


In [ ]:
# Yearly transactions across the full dataset
run_query("""
    SELECT
        CAST(EXTRACT(YEAR FROM date) AS STRING) AS year,
        COUNT(*) AS transactions
    FROM `bigquery-public-data.iowa_liquor_sales.sales`
    GROUP BY year

    UNION ALL

    SELECT
        'Total' AS year,
        COUNT(*) AS transactions
    FROM `bigquery-public-data.iowa_liquor_sales.sales`
    ORDER BY year
""")


,year,transactions
0,2012,2082059
1,2013,2063763
2,2014,2097796
3,2015,2184483
4,2016,2279893
5,2017,2291276
6,2018,2355558
7,2019,2380345
8,2020,2614365
9,2021,2622712


---
## 2. Sales Overview

Revenue distribution across cities, monthly trends, and sales patterns.

In [5]:
run_sql('sales_overview/returns_vs_sales_analysis.sql')

,sale_type,num_transactions,total_dollars,total_bottles,avg_dollars,avg_bottles,earliest,latest
0,Negative (Return?),10415,"-1,597,330.57","-112,389.00",-153.37,-10.79,2022-07-20,2026-02-27
1,Positive (Sale),33599772,"4,997,867,856.91","370,047,895.00",148.75,11.01,2012-01-03,2026-02-27
2,Zero,4943,0.00,"22,417.00",0.00,4.54,2012-02-01,2018-07-30


In [8]:
# Total sales per city (Top 20)
run_sql('sales_overview/city_sales_ranking.sql')

,city,total_transactions,total_sales,avg_sale,total_bottles
0,DES MOINES,335590,"83,556,836.04",248.98,6098911
1,CEDAR RAPIDS,276072,"46,096,295.31",166.97,3346250
2,DAVENPORT,168528,"33,075,821.21",196.26,2679628
3,WEST DES MOINES,138366,"31,855,917.65",230.23,1798024
4,COUNCIL BLUFFS,117497,"25,324,728.85",215.54,1916526
5,SIOUX CITY,114929,"23,310,231.36",202.82,1680814
6,ANKENY,102839,"22,463,576.72",218.43,1320056
7,IOWA CITY,98630,"20,936,433.82",212.27,1463856
8,WATERLOO,119167,"19,963,192.37",167.52,1717564
9,DUBUQUE,102411,"18,053,261.04",176.28,1338193


In [9]:
# Monthly sales summary
run_sql('sales_overview/monthly_sales_summary.sql')

,sale_year,sale_month,num_transactions,total_revenue,avg_transaction_value
0,2024,7,222338,"37,563,557.73",168.95
1,2024,8,214026,"36,529,981.19",170.68
2,2024,9,200540,"34,410,490.22",171.59
3,2024,10,223763,"42,691,915.20",190.79
4,2024,11,217084,"40,090,042.22",184.68
5,2024,12,244498,"41,196,967.68",168.50
6,2025,1,200509,"31,111,677.66",155.16
7,2025,2,176352,"31,308,695.75",177.54
8,2025,3,195133,"33,518,798.61",171.77
9,2025,4,211280,"35,499,986.35",168.02


In [14]:
# Cities with >$1M in sales (HAVING)
run_sql('sales_overview/high_revenue_cities.sql')

,city,total_sales,num_stores
0,DES MOINES,"83,556,836.04",96
1,CEDAR RAPIDS,"46,096,295.31",96
2,DAVENPORT,"33,075,821.21",65
3,WEST DES MOINES,"31,855,917.65",51
4,COUNCIL BLUFFS,"25,324,728.85",51
...,...,...,...
94,JEFFERSON,"1,084,405.05",4
95,CLEARLAKE,"1,058,178.15",4
96,EVANSDALE,"1,054,358.25",6
97,GLENWOOD,"1,052,118.09",8


In [ ]:
# Pivot monthly sales into columns
run_sql('sales_overview/monthly_sales_pivot.sql').head(10)

,city,jul_sales,aug_sales,sep_sales,oct_sales,nov_sales,dec_sales
0,DES MOINES,"4,126,751.79","4,355,382.92","4,100,615.54","4,714,142.11","4,053,837.17","4,849,004.01"
1,CEDAR RAPIDS,"2,500,727.63","2,273,281.23","2,148,229.29","2,745,580.67","2,469,579.83","2,733,711.54"
2,WEST DES MOINES,"1,598,822.29","1,619,865.91","1,487,064.66","1,828,224.17","1,824,540.89","2,218,190.29"
3,DAVENPORT,"1,771,057.09","1,579,803.52","1,674,054.09","1,959,114.66","1,763,288.98","2,026,114.32"
4,COUNCIL BLUFFS,"1,319,222.18","1,384,928.41","1,048,950.67","1,383,736.33","1,387,501.30","1,401,309.96"
5,SIOUX CITY,"1,261,437.33","1,238,480.89","1,053,822.66","1,485,333.81","1,317,400.26","1,360,201.77"
6,ANKENY,"1,112,525.19","1,099,461.55","1,177,393.94","1,360,643.29","1,234,655.42","1,287,319.52"
7,WATERLOO,"1,049,392.32","1,071,518.28","994,850.95","1,236,994.84","1,025,347.00","1,204,722.30"
8,IOWA CITY,"1,159,710.89","1,059,357.26","1,070,080.74","1,305,451.52","1,023,861.22","1,047,804.96"
9,CORALVILLE,"698,536.77","699,258.40","726,872.23","763,386.10","839,343.04","957,936.49"


In [ ]:
# Weekend vs weekday patterns
run_sql('sales_overview/weekend_vs_weekday_patterns.sql').head(10)

---
## 3. Store Performance

Store-level analysis: rankings, segmentation, and benchmarking against averages.

In [96]:
# Store sales tier classification
run_sql('store_performance/store_tier_classification.sql').head(10)

,store_number,store_name,city,total_sales,store_tier,tier_rank
0,2633,HY-VEE #3 / BDI / DES MOINES,DES MOINES,"22,659,090.06",Enterprise,1
1,4829,CENTRAL CITY 2,DES MOINES,"21,730,796.28",Enterprise,1
2,5916,ANOTHER ROUND / DEWITT,DEWITT,"10,367,117.10",Enterprise,1
3,2512,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,IOWA CITY,"9,373,892.73",Enterprise,1
4,3773,BENZ DISTRIBUTING,CEDAR RAPIDS,"8,449,297.68",Enterprise,1
5,6242,WALL TO WALL WINE AND SPIRITS / WEST DES MOINES,WEST DES MOINES,"7,517,690.94",Enterprise,1
6,4312,I-80 LIQUOR / COUNCIL BLUFFS,COUNCIL BLUFFS,"6,651,300.91",Enterprise,1
7,3814,COSTCO WHOLESALE #788 / WDM,WEST DES MOINES,"5,700,443.55",Enterprise,1
8,2614,HY-VEE #3 FOOD & DRUGSTORE / DAVENPORT,DAVENPORT,"5,491,851.89",Enterprise,1
9,5102,WILKIE LIQUORS,MOUNT VERNON,"5,252,442.38",Enterprise,1


In [17]:
# Average bottles per transaction by store
run_sql('store_performance/avg_bottles_per_transaction.sql')

,store_number,store_name,num_transactions,avg_bottles_per_txn,total_sales
0,3814,COSTCO WHOLESALE #788 / WDM,1959,119.61,"5,700,443.55"
1,4677,COSTCO WHOLESALE #1111 / CORALVILLE,1532,116.24,"4,124,037.56"
2,5666,COSTCO WHOLESALE #1325 / DAVENPORT,1304,104.46,"3,199,485.13"
3,10074,COSTCO WHOLESALE #1630 / ANKENY,1420,102.99,"3,487,739.06"
4,4849,C FRESH MARKET,440,53.19,"219,633.65"
5,3477,SAM'S CLUB 6472 / COUNCIL BLUFFS,3151,50.38,"3,277,713.38"
6,3447,SAM'S CLUB 6432 / SIOUX CITY,3802,49.92,"3,686,660.05"
7,3354,SAM'S CLUB 8238 / DAVENPORT,3622,49.40,"3,541,720.10"
8,3420,SAM'S CLUB 6344 / WINDSOR HEIGHTS,4356,49.38,"4,467,983.61"
9,3385,SAM'S CLUB 8162 / CEDAR RAPIDS,4840,46.55,"4,676,429.87"


In [84]:
# Stores outperforming county avg by 50%+
run_sql('store_performance/stores_above_county_avg.sql')

,store_name,county,store_sales,avg_county_sales,pct_above_avg
0,HY-VEE #3 / BDI / DES MOINES,POLK,"22,659,090.06","470,330.28","4,717.70"
1,CENTRAL CITY 2,POLK,"21,730,796.28","470,330.28","4,520.33"
2,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,JOHNSON,"9,373,892.73","392,564.66","2,287.86"
3,BENZ DISTRIBUTING,LINN,"8,449,297.68","388,104.54","2,077.07"
4,ANOTHER ROUND / DEWITT,CLINTON,"10,367,117.10","559,453.47","1,753.08"
5,I-80 LIQUOR / COUNCIL BLUFFS,POTTAWATTAMIE,"6,651,300.91","374,526.38","1,675.92"
6,WALL TO WALL WINE AND SPIRITS / WEST DES MOINES,DALLAS,"7,517,690.94","426,939.44","1,660.83"
7,HY-VEE #2 (1018) / AMES,STORY,"3,975,464.18","266,110.47","1,393.91"
8,WILKIE LIQUORS,LINN,"5,252,442.38","388,104.54","1,253.36"
9,COSTCO WHOLESALE #788 / WDM,DALLAS,"5,700,443.55","426,939.44","1,235.19"


In [89]:
# Each store's best-selling product
run_sql('store_performance/best_product_per_store.sql').head(10)

,store_number,store_name,item_description,total_item_sales
0,2633,HY-VEE #3 / BDI / DES MOINES,TITOS HANDMADE VODKA,"2,233,902.96"
1,4829,CENTRAL CITY 2,TITOS HANDMADE VODKA,"1,917,708.96"
2,5916,ANOTHER ROUND / DEWITT,TITOS HANDMADE VODKA,"1,055,710.68"
3,2512,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,TITOS HANDMADE VODKA,"765,330.84"
4,3773,BENZ DISTRIBUTING,TITOS HANDMADE VODKA,"695,278.93"
5,3814,COSTCO WHOLESALE #788 / WDM,KIRKLAND SIGNATURE AMERICAN VODKA,"682,052.40"
6,4312,I-80 LIQUOR / COUNCIL BLUFFS,TITOS HANDMADE VODKA,"533,709.00"
7,5102,WILKIE LIQUORS,TITOS HANDMADE VODKA,"530,840.88"
8,4677,COSTCO WHOLESALE #1111 / CORALVILLE,KIRKLAND SIGNATURE AMERICAN VODKA,"522,201.60"
9,5666,COSTCO WHOLESALE #1325 / DAVENPORT,KIRKLAND SIGNATURE AMERICAN VODKA,"418,543.20"


In [92]:
# Stores above median sales
run_sql('store_performance/stores_above_median_sales.sql').head(10)

,store_number,store_name,total_sales
0,2633,HY-VEE #3 / BDI / DES MOINES,"22,659,090.06"
1,4829,CENTRAL CITY 2,"21,730,796.28"
2,5916,ANOTHER ROUND / DEWITT,"10,367,117.10"
3,2512,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,"9,373,892.73"
4,3773,BENZ DISTRIBUTING,"8,449,297.68"
5,6242,WALL TO WALL WINE AND SPIRITS / WEST DES MOINES,"7,517,690.94"
6,4312,I-80 LIQUOR / COUNCIL BLUFFS,"6,651,300.91"
7,3814,COSTCO WHOLESALE #788 / WDM,"5,700,443.55"
8,2614,HY-VEE #3 FOOD & DRUGSTORE / DAVENPORT,"5,491,851.89"
9,5102,WILKIE LIQUORS,"5,252,442.38"


In [30]:
# Store sales vs city average (INNER JOIN)
run_sql('store_performance/store_vs_city_avg_comparison.sql').head(5)

,store_number,store_name,city,store_total_sales,city_avg_sales,stores_in_city,diff_from_city_avg
0,2633,HY-VEE #3 / BDI / DES MOINES,DES MOINES,"22,659,090.06","795,779.39",105,"21,863,310.67"
1,4829,CENTRAL CITY 2,DES MOINES,"21,730,796.28","795,779.39",105,"20,935,016.89"
2,2512,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,IOWA CITY,"9,373,892.73","427,274.16",49,"8,946,618.57"
3,5916,ANOTHER ROUND / DEWITT,DEWITT,"10,367,117.10","1,905,465.36",6,"8,461,651.74"
4,3773,BENZ DISTRIBUTING,CEDAR RAPIDS,"8,449,297.68","460,962.95",100,"7,988,334.73"


In [19]:
# All stores with top-selling category (LEFT JOIN)
run_sql('store_performance/top_category_per_store.sql')

,store_number,store_name,city,top_category,category_sales
0,2633,HY-VEE #3 / BDI / DES MOINES,DES MOINES,AMERICAN VODKAS,"3,120,505.33"
1,4829,CENTRAL CITY 2,DES MOINES,100% AGAVE TEQUILA,"3,020,884.20"
2,2512,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,IOWA CITY,AMERICAN VODKAS,"1,669,405.29"
3,6242,WALL TO WALL WINE AND SPIRITS / WEST DES MOINES,WEST DES MOINES,STRAIGHT BOURBON WHISKIES,"1,574,521.14"
4,5916,ANOTHER ROUND / DEWITT,DEWITT,AMERICAN VODKAS,"1,404,221.68"
5,3814,COSTCO WHOLESALE #788 / WDM,WEST DES MOINES,AMERICAN VODKAS,"1,072,274.40"
6,3773,BENZ DISTRIBUTING,CEDAR RAPIDS,100% AGAVE TEQUILA,"1,008,898.68"
7,4312,I-80 LIQUOR / COUNCIL BLUFFS,COUNCIL BLUFFS,CANADIAN WHISKIES,"989,252.87"
8,2647,HY-VEE #7 / CEDAR RAPIDS,CEDAR RAPIDS,STRAIGHT BOURBON WHISKIES,"809,831.71"
9,2663,HY-VEE FOOD STORE / URBANDALE,URBANDALE,STRAIGHT BOURBON WHISKIES,"801,483.92"


In [54]:
# NTILE - store quartiles
run_sql('store_performance/store_quartile_distribution.sql')

,store_number,store_name,city,total_sales,quartile,performance_tier
0,2633,HY-VEE #3 / BDI / DES MOINES,DES MOINES,"22,659,090.06",1,Top 25%
1,4829,CENTRAL CITY 2,DES MOINES,"21,730,796.28",1,Top 25%
2,5916,ANOTHER ROUND / DEWITT,DEWITT,"10,367,117.10",1,Top 25%
3,2512,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,IOWA CITY,"9,373,892.73",1,Top 25%
4,3773,BENZ DISTRIBUTING,CEDAR RAPIDS,"8,449,297.68",1,Top 25%
5,6242,WALL TO WALL WINE AND SPIRITS / WEST DES MOINES,WEST DES MOINES,"7,517,690.94",1,Top 25%
6,4312,I-80 LIQUOR / COUNCIL BLUFFS,COUNCIL BLUFFS,"6,651,300.91",1,Top 25%
7,3814,COSTCO WHOLESALE #788 / WDM,WEST DES MOINES,"5,700,443.55",1,Top 25%
8,2614,HY-VEE #3 FOOD & DRUGSTORE / DAVENPORT,DAVENPORT,"5,491,851.89",1,Top 25%
9,5102,WILKIE LIQUORS,MOUNT VERNON,"5,252,442.38",1,Top 25%


In [52]:
# RANK vs DENSE_RANK vs ROW_NUMBER - side by side
run_sql('store_performance/store_ranking_by_city.sql')

,city,store_name,total_sales,row_num,rank_val,dense_rank_val
0,CEDAR RAPIDS,BENZ DISTRIBUTING,"8,449,297.68",1,1,1
1,CEDAR RAPIDS,SAM'S CLUB 8162 / CEDAR RAPIDS,"4,676,429.87",2,2,2
2,CEDAR RAPIDS,HY-VEE #7 / CEDAR RAPIDS,"4,089,802.27",3,3,3
3,CEDAR RAPIDS,HY-VEE FOOD STORE #5 / CEDAR RAPIDS,"2,977,576.26",4,4,4
4,CEDAR RAPIDS,HY-VEE FOOD STORE #3 (1056) / CEDAR RAPIDS,"1,703,272.70",5,5,5
...,...,...,...,...,...,...
200,DES MOINES,WALGREENS #05362 / DES MOINES,"4,384.62",101,101,101
201,DES MOINES,C FRESH MARKET / DES MOINES,"3,553.80",102,102,102
202,DES MOINES,WALGREENS #05721 / DES MOINES,"2,221.38",103,103,103
203,DES MOINES,WALGREENS #05777 / DES MOINES,"1,013.36",104,104,104


In [56]:
# PERCENT_RANK and CUME_DIST
run_sql('store_performance/store_percentile_ranking.sql')

,store_name,total_sales,percentile,cumulative_dist
0,HY-VEE #3 / BDI / DES MOINES,"22,659,090.06",100.00,100.00
1,CENTRAL CITY 2,"21,730,796.28",99.96,99.96
2,ANOTHER ROUND / DEWITT,"10,367,117.10",99.92,99.92
3,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,"9,373,892.73",99.88,99.88
4,BENZ DISTRIBUTING,"8,449,297.68",99.84,99.84
5,WALL TO WALL WINE AND SPIRITS / WEST DES MOINES,"7,517,690.94",99.79,99.79
6,I-80 LIQUOR / COUNCIL BLUFFS,"6,651,300.91",99.75,99.75
7,COSTCO WHOLESALE #788 / WDM,"5,700,443.55",99.71,99.71
8,HY-VEE #3 FOOD & DRUGSTORE / DAVENPORT,"5,491,851.89",99.67,99.67
9,WILKIE LIQUORS,"5,252,442.38",99.63,99.63


In [44]:
# Store contribution % within city
run_sql('store_performance/store_city_contribution_pct.sql').head(5)

,store_number,store_name,city,store_total,city_total,pct_of_city
0,10206,HAWKEYE SMOKE SHOP / FAIRFIELD,None,"154,982.77","330,441.04",46.90
1,2201,HAPPY'S WINE & SPIRITS WHOLESALE,None,"92,237.40","330,441.04",27.91
2,2619,HY-VEE WINE AND SPIRITS / WDM,None,"32,768.18","330,441.04",9.92
3,5545,CASEY'S GENERAL STORE #91 / DALLAS CENTER,None,"12,166.98","330,441.04",3.68
4,5181,CASEY'S GENERAL STORE #3434 / DENV,None,"7,107.90","330,441.04",2.15


---
## 4. Product & Category Analysis

Category performance, price segmentation, and top products.

In [ ]:
# Price segmentation
run_sql('product_category/product_price_segmentation.sql').head(10)

In [15]:
# Category performance
run_sql('product_category/category_sales_performance.sql')

,category_name,transactions,total_bottles,total_revenue,avg_bottle_price
0,AMERICAN VODKAS,649941,10725783,"108,955,547.21",10.58
1,CANADIAN WHISKIES,382909,4678664,"77,784,371.65",17.70
2,STRAIGHT BOURBON WHISKIES,332428,2854369,"66,933,635.26",27.11
3,100% AGAVE TEQUILA,233051,1947289,"55,186,101.69",33.52
4,WHISKEY LIQUEUR,288903,7785927,"44,727,590.18",16.87
5,SPICED RUM,146800,2188607,"34,707,352.64",14.56
6,TENNESSEE WHISKIES,124844,1204155,"27,166,452.63",22.53
7,TEMPORARY & SPECIALTY PACKAGES,128618,898363,"26,156,215.18",46.24
8,IMPORTED VODKAS,91129,1237618,"19,888,868.85",17.98
9,IMPORTED CORDIALS & LIQUEURS,95296,804042,"19,061,885.44",25.22


In [53]:
# Top 3 products per category
run_sql('product_category/top_products_per_category.sql')

,category_name,item_description,total_bottles,total_revenue,product_rank
0,100% AGAVE TEQUILA,PATRON SILVER,251114,"8,349,114.13",1
1,100% AGAVE TEQUILA,DON JULIO REPOSADO,113690,"4,432,716.85",2
2,100% AGAVE TEQUILA,DON JULIO BLANCO,81527,"3,247,390.48",3
3,AGED DARK RUM,BUMBU RUM,13274,"353,718.50",1
4,AGED DARK RUM,MYERS'S ORIGINAL DARK RUM,5321,"106,313.58",2
...,...,...,...,...,...
126,WHISKEY LIQUEUR,FIREBALL CINNAMON WHISKEY MINI SLEEVE,1428795,"9,444,249.69",2
127,WHISKEY LIQUEUR,FIREBALL CINNAMON WHISKEY PET,305006,"5,160,094.50",3
128,WHITE RUM,BACARDI SUPERIOR,297170,"4,077,215.10",1
129,WHITE RUM,BACARDI SUPERIOR PET,74268,"1,457,257.42",2


In [35]:
# Category vs overall average (CROSS JOIN)
run_sql('product_category/category_vs_overall_avg.sql')

,category_name,cat_avg_sale,overall_avg,diff_from_overall,cat_transactions
0,SPECIAL ORDER ITEMS,622.43,171.76,450.67,2498
1,IRISH WHISKIES,303.33,171.76,131.57,50545
2,IMPORTED DISTILLED SPIRITS SPECIALTY,252.94,171.76,81.18,18251
3,FLAVORED GIN,240.52,171.76,68.76,8581
4,IMPORTED BRANDIES,239.90,171.76,68.14,57834
5,100% AGAVE TEQUILA,236.80,171.76,65.04,233051
6,SPICED RUM,236.43,171.76,64.67,146800
7,SINGLE MALT SCOTCH,230.71,171.76,58.95,24443
8,IMPORTED VODKAS,218.25,171.76,46.49,91129
9,TENNESSEE WHISKIES,217.60,171.76,45.84,124844


In [46]:
# Best/worst month per category
run_sql('product_category/category_best_worst_months.sql')

,category_name,best_month,best_month_sales,worst_month,worst_month_sales
0,AMERICAN VODKAS,2024-08-01,"6,909,632.84",2024-09-01,"4,570,830.71"
1,CANADIAN WHISKIES,2024-10-01,"5,559,409.22",2025-11-01,"2,856,174.13"
2,STRAIGHT BOURBON WHISKIES,2025-12-01,"4,563,227.48",2026-01-01,"2,544,522.18"
3,SPICED RUM,2024-11-01,"3,502,759.01",2026-02-01,"1,109,797.50"
4,100% AGAVE TEQUILA,2025-04-01,"3,225,682.83",2026-02-01,"2,111,230.81"
5,TEMPORARY & SPECIALTY PACKAGES,2024-11-01,"3,119,502.81",2026-02-01,"454,032.87"
6,WHISKEY LIQUEUR,2025-09-01,"2,639,751.93",2025-02-01,"1,774,634.97"
7,TENNESSEE WHISKIES,2024-10-01,"1,745,719.88",2025-01-01,"1,015,284.95"
8,IMPORTED CORDIALS & LIQUEURS,2025-12-01,"1,284,073.84",2026-01-01,"738,635.97"
9,IMPORTED VODKAS,2025-10-01,"1,247,874.08",2026-02-01,"767,442.06"


---
## 5. Time Series & Trends

Temporal patterns: moving averages, month-over-month growth, cumulative totals, and anomaly detection.

In [48]:
# 3-month and 6-month SMA
run_sql('time_series/moving_avg_3_6_month.sql').head(5)

,sale_month,total_sales,sma_3_month,sma_6_month
0,2024-01-01,"32,972,812.84","32,972,812.84","32,972,812.84"
1,2024-02-01,"35,101,475.45","34,037,144.15","34,037,144.15"
2,2024-03-01,"34,164,169.29","34,079,485.86","34,079,485.86"
3,2024-04-01,"36,385,547.87","35,217,064.20","34,656,001.36"
4,2024-05-01,"39,359,095.65","36,636,270.94","35,596,620.22"


In [49]:
# 4-week SMA per city
run_sql('time_series/weekly_moving_avg_per_city.sql').head(5)

,city,sale_week,weekly_sales,sma_4_week
0,CEDAR RAPIDS,2024-06-30,"661,941.02","661,941.02"
1,CEDAR RAPIDS,2024-07-07,"424,037.98","542,989.50"
2,CEDAR RAPIDS,2024-07-14,"474,187.87","520,055.62"
3,CEDAR RAPIDS,2024-07-21,"435,926.73","499,023.40"
4,CEDAR RAPIDS,2024-07-28,"556,626.43","472,694.75"


In [50]:
# Moving average with anomaly detection
run_sql('time_series/sales_anomaly_detection.sql').head(5)

,sale_month,total_sales,sma_3,std_3,anomaly_flag
0,2024-01-01,"32,972,812.84","32,972,812.84",NaN,NORMAL
1,2024-02-01,"35,101,475.45","34,037,144.15","1,505,191.77",NORMAL
2,2024-03-01,"34,164,169.29","34,079,485.86","1,066,855.00",NORMAL
3,2024-04-01,"36,385,547.87","35,217,064.20","1,115,191.14",NORMAL
4,2024-05-01,"39,359,095.65","36,636,270.94","2,606,522.88",NORMAL


In [51]:
# Category trends with SMA
run_sql('time_series/category_trend_comparison.sql').head(5)

,category_name,sale_month,monthly_sales,sma_3_month,deviation_from_sma
0,AMERICAN VODKAS,2024-01-01,"5,156,783.18","5,156,783.18",0.00
1,AMERICAN VODKAS,2024-02-01,"5,318,724.41","5,237,753.79","80,970.62"
2,AMERICAN VODKAS,2024-03-01,"4,851,894.16","5,109,133.92","-257,239.76"
3,AMERICAN VODKAS,2024-04-01,"5,409,313.60","5,193,310.72","216,002.88"
4,AMERICAN VODKAS,2024-05-01,"6,316,239.04","5,525,815.60","790,423.44"


In [34]:
# Month-over-month comparison (Self-Join)
run_sql('time_series/month_over_month_by_store.sql').head(5)

,store_number,store_name,current_year,previous_year,current_month,previous_month,current_sales,previous_month_sales,mom_change,mom_pct_change
0,2633,HY-VEE #3 / BDI / DES MOINES,2024,2024,10,9,"1,384,790.33","1,039,005.97","345,784.36",33.28
1,2633,HY-VEE #3 / BDI / DES MOINES,2025,2025,10,9,"1,367,842.33","1,025,843.60","341,998.73",33.34
2,2633,HY-VEE #3 / BDI / DES MOINES,2025,2025,7,6,"1,338,895.80","1,256,226.41","82,669.39",6.58
3,2633,HY-VEE #3 / BDI / DES MOINES,2025,2025,4,3,"1,324,128.75","1,097,886.94","226,241.81",20.61
4,4829,CENTRAL CITY 2,2025,2025,6,5,"1,304,502.10","1,245,130.35","59,371.75",4.77


In [45]:
# LAG/LEAD - month-over-month comparison
run_sql('time_series/month_over_month_growth.sql')

,sale_month,total_sales,prev_month_sales,next_month_sales,mom_change,mom_pct_change
0,2024-07-01,"37,563,557.73",NaN,"36,529,981.19",NaN,NaN
1,2024-08-01,"36,529,981.19","37,563,557.73","34,410,490.22","-1,033,576.54",-2.75
2,2024-09-01,"34,410,490.22","36,529,981.19","42,691,915.20","-2,119,490.97",-5.80
3,2024-10-01,"42,691,915.20","34,410,490.22","40,090,042.22","8,281,424.98",24.07
4,2024-11-01,"40,090,042.22","42,691,915.20","41,196,967.68","-2,601,872.98",-6.09
5,2024-12-01,"41,196,967.68","40,090,042.22","31,111,677.66","1,106,925.46",2.76
6,2025-01-01,"31,111,677.66","41,196,967.68","31,308,695.75","-10,085,290.02",-24.48
7,2025-02-01,"31,308,695.75","31,111,677.66","33,518,798.61","197,018.09",0.63
8,2025-03-01,"33,518,798.61","31,308,695.75","35,499,986.35","2,210,102.86",7.06
9,2025-04-01,"35,499,986.35","33,518,798.61","35,589,683.49","1,981,187.74",5.91


In [43]:
# Cumulative sales by city (running total)
run_sql('time_series/cumulative_sales_per_city.sql').head(5)

,city,sale_month,monthly_sales,cumulative_sales,rn
0,CEDAR RAPIDS,2024-07-01,"2,500,727.63","2,500,727.63",5
1,CEDAR RAPIDS,2024-08-01,"2,273,281.23","4,774,008.86",11
2,CEDAR RAPIDS,2024-09-01,"2,148,229.29","6,922,238.15",14
3,CEDAR RAPIDS,2024-10-01,"2,745,580.67","9,667,818.82",2
4,CEDAR RAPIDS,2024-11-01,"2,469,579.83","12,137,398.65",6


In [55]:
# Best-selling day per city
run_sql('time_series/best_selling_day_per_city.sql')

,city,date,daily_sales
0,URBANDALE,2025-10-07,"778,573.64"
1,DES MOINES,2025-04-21,"598,814.42"
2,CEDAR RAPIDS,2024-11-06,"372,750.83"
3,DAVENPORT,2024-11-05,"328,205.12"
4,SIOUX CITY,2025-02-06,"301,783.02"
5,WEST DES MOINES,2024-12-02,"289,333.20"
6,DUBUQUE,2025-11-07,"286,384.21"
7,ANKENY,2024-11-20,"282,375.28"
8,CEDAR FALLS,2024-10-10,"268,381.89"
9,CORALVILLE,2024-12-17,"256,899.17"


In [93]:
# Growing counties with high store count (multi-CTE funnel)
run_sql('time_series/growing_counties_analysis.sql').head(10)

,county,avg_monthly,growth_pct,num_stores
0,DECATUR,"23,705.28",27.83,5
1,WAYNE,"23,285.26",22.42,7
2,EMMET,"66,433.62",15.34,7
3,HAMILTON,"101,938.61",11.92,15
4,JEFFERSON,"126,016.93",6.73,11
5,CHEROKEE,"86,252.92",5.98,9
6,WASHINGTON,"214,294.73",5.63,22
7,TAYLOR,"13,427.10",5.18,5
8,STORY,"984,608.75",2.19,65
9,PAGE,"132,688.58",2.02,12


In [94]:
# EXISTS - stores selling premium products
run_sql('time_series/premium_product_stores.sql')

,store_number,store_name,city
0,5876,'DA BOOZE BARN / WEST BEND,WEST BEND
1,5443,1ST STOP BEVERAGE SHOP,DES MOINES
2,5736,218 FUEL EXPRESS,FLOYD
3,5736,218 FUEL EXPRESS / FLOYD,FLOYD
4,4787,380BP / SWISHER,SWISHER
5,10223,4 WAY STOP SHOP / MOVILLE,MOVILLE
6,5498,6 CORNERS GAS & GRUB,ARLINGTON
7,10219,7 DAYS MART / COUNCIL BLUFFS,COUNCIL BLUFFS
8,3881,7 RAYOS LIQUOR STORE,MARSHALLTOWN
9,10635,7 STAR LIQUOR & TOBACCO / WATERLOO,WATERLOO


---
## 6. Data Quality

Data profiling: duplicates, NULL handling, and deduplication strategies.

In [ ]:
# NULL handling with COALESCE
run_sql('data_quality/null_handling_cleanup.sql').head(10)

In [73]:
# Find duplicates
run_sql('data_quality/identify_duplicates.sql').head()

,store_number,date,item_description,sale_dollars,occurrence_count
0,5678,2024-12-03,GREY GOOSE,405.00,20
1,5444,2025-10-07,GRANGALA TRIPLE ORANGE LIQUEUR,207.00,14
2,5444,2025-02-18,GRANGALA TRIPLE ORANGE LIQUEUR,207.00,11
3,10303,2025-12-28,173 CRAFT DISTILLERY HONEY LEMON FLAVORED GIN,162.00,10
4,5444,2026-02-10,GRANGALA TRIPLE ORANGE LIQUEUR,207.00,9


In [76]:
# ROW_NUMBER deduplication pattern
run_sql('data_quality/deduplicate_transactions.sql').head()

,invoice_and_item_number,date,store_number,store_name,address,city,zip_code,store_location,county_number,county,...,item_description,pack,bottle_volume_ml,state_bottle_cost,state_bottle_retail,bottles_sold,sale_dollars,volume_sold_liters,volume_sold_gallons,rn
0,INV-74817200014,2024-10-01,10003,TOP SHELF LIQUOR / IOWA CITY,412 HIGHWAY 1,IOWA CITY,52246.0,POINT(-91.54362 41.64755),None,JOHNSON,...,DOC HOLLIDAY BOURBON 7 YEARS,6,750,55.00,82.50,1,82.50,0.75,0.19,1
1,INV-74817200022,2024-10-01,10003,TOP SHELF LIQUOR / IOWA CITY,412 HIGHWAY 1,IOWA CITY,52246.0,POINT(-91.54362 41.64755),None,JOHNSON,...,MICHTERS US*1 SOUR MASH WHISKEY,6,750,24.50,36.75,2,75.00,1.50,0.39,1
2,INV-75057800024,2024-10-08,10003,TOP SHELF LIQUOR / IOWA CITY,412 HIGHWAY 1,IOWA CITY,52246.0,POINT(-91.54362 41.64755),None,JOHNSON,...,CARAVELLA LIMONCELLO,6,750,11.75,17.63,1,17.63,0.75,0.19,1
3,INV-75057800034,2024-10-08,10003,TOP SHELF LIQUOR / IOWA CITY,412 HIGHWAY 1,IOWA CITY,52246.0,POINT(-91.54362 41.64755),None,JOHNSON,...,JOSE CUERVO ESPECIAL SILVER,12,750,12.00,18.00,2,36.00,1.50,0.39,1
4,INV-75613400032,2024-10-22,10003,TOP SHELF LIQUOR / IOWA CITY,412 HIGHWAY 1,IOWA CITY,52246.0,POINT(-91.54362 41.64755),None,JOHNSON,...,PARAMOUNT PEPPERMINT SCHNAPPS,12,750,4.25,6.38,4,25.52,3.00,0.79,1


In [78]:
# DISTINCT vs GROUP BY
run_sql('data_quality/unique_store_item_combos.sql').head(5)

,city,county
0,None,None
1,ACKLEY,HARDIN
2,ADAIR,ADAIR
3,ADEL,DALLAS
4,AFTON,UNION


In [79]:
# Keep latest record per store
run_sql('data_quality/latest_record_per_store.sql').head(5)

,store_number,store_name,address,city,zip_code
0,10003,TOP SHELF LIQUOR / IOWA CITY,412 HIGHWAY 1,IOWA CITY,52246.0
1,10004,JACK & JILL STORE / WEST BRANCH,115 EAST MAIN STREET,WEST BRANCH,52358.0
2,10005,THE CROWN LIQUOR / IOWA CITY,324 EAST WASHINGTON STREET,IOWA CITY,52240.0
3,10016,KWIK STAR #1180 / PERRY,511 1ST AVE,PERRY,50220.0
4,10017,KWIK STAR #1119 / EMMETSBURG,1303 MAIN ST,EMMETSBURG,50536.0


In [83]:
# Data quality - duplicate count summary
run_sql('data_quality/duplicate_count_summary.sql').head(10)

,total_rows,unique_rows,duplicate_rows,dup_percentage
0,4170032,3797235,372797,8.94


---
## SQL Quick Reference

| Concept | Key Syntax | When to Use |
|---------|-----------|-------------|
| GROUP BY | `GROUP BY col HAVING condition` | Aggregate data by categories |
| INNER JOIN | `A JOIN B ON A.key = B.key` | Only matching rows from both tables |
| LEFT JOIN | `A LEFT JOIN B ON ...` | All rows from A, matching from B |
| ROW_NUMBER | `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` | Dedup, pagination, top-N |
| RANK | `RANK() OVER (...)` | Ranking with gaps after ties |
| DENSE_RANK | `DENSE_RANK() OVER (...)` | Ranking without gaps |
| Moving Avg | `AVG() OVER (ORDER BY ... ROWS BETWEEN N PRECEDING AND CURRENT ROW)` | Smooth time series |
| LAG/LEAD | `LAG(col, 1) OVER (ORDER BY ...)` | Compare to previous/next row |
| CTE | `WITH name AS (SELECT ...)` | Break complex queries into steps |
| CASE WHEN | `CASE WHEN cond THEN val ELSE val END` | Conditional logic in SQL |
| Dedup | `ROW_NUMBER() ... WHERE rn = 1` | Keep first/latest occurrence |
